# 06 · Heterogeneous evidence graph and deployable service

Keep ignition hypotheses distinct from spreading physics. A line near a candidate or lightning event prior to ignition is a spatial/temporal association, never an established cause.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / "src").exists(): ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import json
import matplotlib.pyplot as plt
INPUT = ROOT / "data/synthetic/toy_firms.csv"  # SYNTHETIC DEMO, NOT historical NASA data
WEATHER = ROOT / "data/synthetic/toy_weather.json"
assert INPUT.exists(), "Run `python scripts/make_synthetic.py` from repository root first"
print("Data mode: SYNTHETIC - demonstration only")

In [ ]:
from wildfire.service import ApplicationService,ServerSettings
from wildfire.evidence import build_evidence_graph
from wildfire.ingest import load_firms,load_weather
from wildfire.inference import infer_origin,InferenceOptions
sat=load_firms(INPUT);wx=load_weather(WEATHER)
model=infer_origin(sat,wx,InferenceOptions(ensemble=False))
evidence=build_evidence_graph(model,
                  str(ROOT/'data/synthetic/toy_powerlines.geojson'),
                  str(ROOT/'data/synthetic/toy_lightning.csv'))
print('Entities:',len(evidence['nodes']),'Relations:',len(evidence['edges']))
for e in evidence['edges'][-3:]:print(e)

In [ ]:
# The API service uses the same pure-Python inference engine as the notebooks.
service=ApplicationService(ServerSettings(str(INPUT),str(WEATHER),
    powerlines=str(ROOT/'data/synthetic/toy_powerlines.geojson'),
    lightning=str(ROOT/'data/synthetic/toy_lightning.csv'),mode='synthetic'))
print('Platform / integrity:',json.dumps(service.health(),indent=2))
print('Fresh inference best cell:',service.infer('wind',1200,5,False)['inference']['best_origin'])

In [ ]:
# Export portable graph tables; optionally adapt for TuringDB importer.
import csv
out=ROOT/'artifacts/evidence_entities.csv'
with out.open('w',newline='') as f:
    w=csv.writer(f);w.writerow(['id','kind','properties_json'])
    for node in evidence['nodes']:
        w.writerow([node['id'],node['kind'],json.dumps({k:v for k,v in node.items() if k not in ('id','kind')})])
print('Saved',out)

**IBM Z proof:** run `uname -m`, Python version, `/api/health` and `/api/infer` ON your LinuxONE VM. No s390x TuringDB binary is assumed. The TuringDB graph should run separately, or use portable CSV exports.